# Reproducing Buccino et al. (2020) from dandiset 000034

**Paper.** Buccino AP, Hurwitz CL, Garcia S, Magland J, Siegle JH, Hurwitz R, Hennig MH (2020).
SpikeInterface, a unified framework for spike sorting. *eLife* 9:e61834.
[doi:10.7554/eLife.61834](https://doi.org/10.7554/eLife.61834). The claims file
(`claims/000034/10.7554-elife.61834.yaml`) was written from the version of record (full text from
Europe PMC, PMC7704107, with the figure-supplement legends, the Methods and the author response)
and the paper's notebooks in `SpikeInterface/spikeinterface.github.io` (last changed at ee08fb6).

**Dandiset.** [DANDI:000034](https://dandiarchive.org/dandiset/000034/0.211030.0713), version
0.211030.0713 (doi:10.48324/dandi.000034/0.211030.0713). The paper cites the draft
(`dandiset/000034/draft`), as it was published before any version existed. This is the only
published version; the data check below shows that it holds the same six assets (same asset ids
and sizes) as today's draft, so the choice does not change the data.

**What the dandiset holds.** The four recordings the paper analyses (mouse Neuropixels, simulated
MEArec Neuropixels with ground truth, rat cortex Neuropixels, mouse retina Biocam) and the two
experts' manual curations of the paper's Kilosort2 output on the mouse recording. It holds no
output of any of the six spike sorters the paper compares.

**Why most claims are blocked.** Every finding except the presence of the recordings rests on
running six spike sorters on these recordings. Kilosort2, IronClust and HDSort are MATLAB
programs (Kilosort2 also needs a CUDA GPU); they cannot be put in a locked Python environment, and
this runner has neither MATLAB nor a GPU. The section "Why the sorter claims are blocked" also
measures how fast the recordings stream: reading any of them whole would take hours, over the
30 minute limit of a notebook here.

| Claim | Tested here | Blocked by |
|---|---|---|
| recordings-as-described (link) | yes | |
| fig1b-unit-counts | no | not_runnable |
| fig1c-low-agreement | no | not_runnable |
| fig1d-most-units-unmatched | no | not_runnable |
| fig1s2-agreement-scores-high | no | not_runnable |
| fig1s3-s4-other-recordings-disagree | no | not_runnable |
| fig2c-sim-unit-counts | no | not_runnable |
| fig2s1a-precision-recall-bias | no | not_runnable |
| fig2s1b-accuracy-snr | no | not_runnable |
| fig2d-ks2-highest-scores | no | not_runnable |
| fig2e-ks2-unit-types | no | not_runnable |
| fig2e-ks2-curated-unit-types | no | not_runnable |
| fig2e-tdc-few-fp-hds-many-fp | no | not_runnable |
| fig3a-sim-low-agreement | no | not_runnable |
| fig3b-sorters-prone-to-low-agreement | no | not_runnable |
| fig3c-fp-found-by-one-sorter | no | not_runnable |
| fig3s1-ks2-ic-best-pair | no | not_runnable |
| fig3s1-pairs-remove-fp-four-keep-tp | no | not_runnable |
| fig3d-snr-rises-with-agreement | no | not_runnable |
| fig3s2-snr-does-not-separate-ks2-fp | no | not_runnable |
| fig4a-curators-agree | in part (the curator counts; blocked as it depends on fig1b-unit-counts) | not_runnable |
| fig4b-single-sorter-match-curated | no | not_runnable |
| fig4c-consensus-match-curated | no | not_runnable |
| fig4d-nonconsensus-rarely-curated | no | not_runnable |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

import si000034 as si

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
started = time.time()
print(f"dandiset {si.DANDISET_ID} version {si.DANDISET_VERSION}")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
pinned = pd.DataFrame(si.list_assets(si.DANDISET_VERSION))
draft = pd.DataFrame(si.list_assets("draft"))
print("Assets of version", si.DANDISET_VERSION)
print(pinned.to_string(index=False))
same = pinned.merge(draft, on=["path", "size", "asset_id"], how="outer", indicator=True)
print("\nAssets in both the pinned version and the draft, same id and size:", int((same._merge == "both").sum()),
      "; only in one:", int((same._merge != "both").sum()))
info = si.version_info()
print("Version name:", info["metadata"]["name"])
print("Description:", info["metadata"]["description"][-250:])

The pinned version and the draft hold the same six files. The description says the dandiset holds
all recordings analysed in the paper.

In [ ]:
files = {"curated 1": si.CURATED_1, "curated 2": si.CURATED_2, **si.RECORDINGS}
handles, rows = {}, []
for name, path in files.items():
    f = si.open_nwb(path)
    handles[name] = f
    d = si.describe(f)
    rows.append({"name": name, "path": path.split("/")[-1], **d})
desc = pd.DataFrame(rows).set_index("name")
cols = ["subject_id", "species", "age", "session_start_time", "frames", "channels", "rate_hz", "duration_s",
        "dtype", "chunks", "conversion", "offset", "unit", "starting_time", "units", "unit_columns"]
print(desc[cols].to_string())

Every recording starts at time 0 and has a regular rate. Each stores its raw data in chunks of one
channel by several thousand frames, so a read of a short stretch of all channels fetches one chunk
per channel. The curated files hold a Units table with spike times only (no rate, no labels, no
link to the Kilosort2 units). The simulated file holds its 250 ground-truth units with their soma
locations. No file says which cells are excitatory or inhibitory, or how the recordings were cut or
filtered (`filtering` is "none" and `location` "unknown" for every channel).

In [ ]:
cur = {k: si.load_units(handles[k]) for k in ["curated 1", "curated 2"]}
gt = si.load_units(handles["MEArec"])
for name, units in [*cur.items(), ("MEArec ground truth", gt)]:
    t = np.concatenate(list(units.values()))
    n = np.array([len(v) for v in units.values()])
    print(f"{name}: {len(units)} units, ids {min(units)} to {max(units)}, {len(t)} spikes, times {t.min():.4f} to {t.max():.4f} s, "
          f"spikes per unit {n.min()} to {n.max()} (median {np.median(n):.0f}), sorted within every unit: "
          f"{all(np.all(np.diff(v) >= 0) for v in units.values())}")
for name in cur:
    t = np.concatenate(list(cur[name].values()))
    print(f"{name}: largest distance of t*30000 from a whole frame: {np.abs(t * 30000 - np.round(t * 30000)).max():.1e} frames")
ids1, ids2 = set(cur["curated 1"]), set(cur["curated 2"])
print("Unit ids in both curations:", len(ids1 & ids2), "; spike trains identical under the same id:",
      sum(np.array_equal(cur["curated 1"][u], cur["curated 2"][u]) for u in ids1 & ids2))
soma = handles["MEArec"]["units/soma_location"][:]
print("MEArec soma locations (um): x", soma[:, 0].min().round(1), "to", soma[:, 0].max().round(1),
      "; y", soma[:, 1].min().round(1), "to", soma[:, 1].max().round(1), "; z", soma[:, 2].min().round(1), "to", soma[:, 2].max().round(1))

The curated spike times lie within the 900 s of the mouse recording and fall on whole frames at
30 kHz, which is consistent with spike times taken from that recording. The two curations share
many unit ids with identical spike trains; these are the Kilosort2 clusters both experts kept
unchanged, as expected of two curations of one sorting. They are two different files, not one
recording deposited twice. The ground-truth spikes lie within the simulated recording's 600 s.

In [ ]:
env = si.environment()
print("Software the paper's sorters need, on this runner:", env)

### Clock and units of the raw data

The curated spike times and the ground-truth spike times are checked against the raw traces they
belong to: the mean trace at each unit's spike frames, shifted by -1.5 to +1.5 ms, should have its
deepest trough at lag 0 if the spike times share the recording's clock, and random frames give the
reference. Only the first chunk of each channel is read (1.76 s of the mouse recording, 0.59 s of
the simulation), because streaming is slow; the time of each read is printed and used below.

In [ ]:
rng_clock = np.random.default_rng(34001)
reads = []
lags = np.arange(-45, 46)

mouse = handles["mouse412804"]
n_mouse = mouse["acquisition/ElectricalSeries/data"].chunks[0] * 2
x, secs = si.read_block(mouse, n_mouse)
reads.append({"recording": "mouse412804", "frames": n_mouse, "channels": x.shape[1], "seconds_of_data": n_mouse / 30000, "read_s": secs})
print(f"mouse412804: {x.shape} {x.dtype}, values {x.min()} to {x.max()} (x conversion 1e-6 = volts, i.e. microvolts), read in {secs:.1f} s")
y = si.bandpass_cmr(x, 30000.0)
print(f"  band-passed: median channel SD {np.median(y.std(0)):.1f} uV")
for name in cur:
    trains = {u: si.to_frames(t) for u, t in cur[name].items()}
    r = si.lag_scan(y, trains, lags, min_spikes=5, rng=rng_clock)
    print(f"  {name}: {len(r['unit'])} units with >= 5 spikes in the block; lag of the deepest trough (frames): median "
          f"{np.median(r['best_lag']):.0f}, 10th to 90th percentile {np.percentile(r['best_lag'], 10):.0f} to "
          f"{np.percentile(r['best_lag'], 90):.0f}, within 3 frames: {np.mean(np.abs(r['best_lag']) <= 3):.2f}; "
          f"trough at lag 0 median {np.median(r['amp_at_0']):.0f} uV, at random frames {np.median(r['amp_random']):.0f} uV")
del x, y

In [ ]:
mea = handles["MEArec"]
n_mea = mea["acquisition/ElectricalSeries/data"].chunks[0]
x, secs = si.read_block(mea, n_mea)
reads.append({"recording": "MEArec", "frames": n_mea, "channels": x.shape[1], "seconds_of_data": n_mea / 32000, "read_s": secs})
print(f"MEArec: {x.shape} {x.dtype}, values {x.min():.1f} to {x.max():.1f} uV, read in {secs:.1f} s")
mad_sd = np.median(np.abs(x - np.median(x, 0)), 0) / 0.6745
print(f"  noise SD per channel from the median absolute deviation: median {np.median(mad_sd):.2f} uV, range "
      f"{mad_sd.min():.2f} to {mad_sd.max():.2f} (paper: additive Gaussian noise of 10 uV SD)")
y = si.bandpass_cmr(x, 32000.0)
trains = {u: si.to_frames(t, 32000.0) for u, t in gt.items()}
r = si.lag_scan(y, trains, lags, min_spikes=2, rng=rng_clock)
print(f"  ground truth: {len(r['unit'])} units with >= 2 spikes in the block; lag of the deepest trough: median "
      f"{np.median(r['best_lag']):.0f}, within 3 frames: {np.mean(np.abs(r['best_lag']) <= 3):.2f}; trough at lag 0 median "
      f"{np.median(r['amp_at_0']):.0f} uV, at random frames {np.median(r['amp_random']):.0f} uV")
del x, y

In [ ]:
rat = handles["rat c1"]
n_rat = rat["acquisition/ElectricalSeries/data"].chunks[0]
x, secs = si.read_block(rat, n_rat, slice(0, 128))
reads.append({"recording": "rat c1", "frames": n_rat, "channels": x.shape[1], "seconds_of_data": n_rat / 30000, "read_s": secs})
print(f"rat c1, channels 0 to 127: {x.dtype}, values {x.min():.1f} to {x.max():.1f}, per-channel mean "
      f"{x.mean(0).min():.1f} to {x.mean(0).max():.1f}, median channel SD {np.median(x.std(0)):.1f}, read in {secs:.1f} s")
ret = handles["retina"]
n_ret = ret["acquisition/ElectricalSeries/data"].chunks[0]
x, secs = si.read_block(ret, n_ret, slice(0, 256))
reads.append({"recording": "retina", "frames": n_ret, "channels": x.shape[1], "seconds_of_data": n_ret / desc.loc["retina", "rate_hz"], "read_s": secs})
vals, counts = np.unique(x, return_counts=True)
print(f"retina, channels 0 to 255: {x.dtype}, values {x.min()} to {x.max()}, median {np.median(x):.0f}, median channel SD "
      f"{np.median(x.std(0)):.1f}, channels constant over the block: {(x.std(0) == 0).sum()}, samples at {x.max()}: "
      f"{counts[-1]}, read in {secs:.1f} s")
print("retina stated conversion", desc.loc["retina", "conversion"], desc.loc["retina", "unit"], "; offset attribute:", desc.loc["retina", "offset"])
del x

The mouse curations and the simulated ground truth share the clocks of their recordings: for most
units the trough of the mean trace is at lag 0 or within a few frames, and much deeper than at
random frames (numbers above). This is shown on the first seconds of each recording only. The
mouse and simulated data in microvolts have plausible ranges. The noise level of the simulation
estimated from its traces is above the paper's 10 uV (printed above), but the estimate includes the
overlapping spikes of 250 cells, so it does not show what noise was added; this is left open. The
rat recording carries a DC offset that differs between channels. The retina recording is stored as
unsigned integers centred near 2048 and reaching 4096, which is consistent with raw ADC counts of a
12-bit converter, not with microvolts; read with the stated conversion (1e-6 V per unit, no
offset) every channel would sit near +2 mV, and samples at 4096 (printed above) look like
saturation, which the paper's notebook removes with `blank_saturation`. The gain and offset that turn these counts into volts
are not in the file. No claim tested here uses the retina's values.

## recordings-as-described (link claim)

> The dandiset holds the recordings the paper analyses and the files its notebooks name, with the
> properties the paper and the notebooks give.

Criterion: each of the five named files is present, and a recording of mouse 412804 with 30 kHz,
15 min (27,000,000 frames) and 246 (paper) or 248 (notebook) channels; the simulated recording
lasts 10 min and its ground truth has 250 units; the rat recording has 384 channels at 30 kHz; the
Biocam recording has 1024 channels at about 23199 Hz.

In [ ]:
named = ["sub-mouse412804_ses-20200824T155542.nwb", "sub-mouse412804_ses-20200824T155543.nwb",
         "sub-MEAREC-250neuron-Neuropixels_ecephys.nwb", "sub-c1_ecephys.nwb", "sub-P29-16-05-14-retina02-left_ecephys.nwb"]
present = set(pinned.path.str.split("/").str[-1])
checks = [(f"file {n} present", n in present, "") for n in named]
m, s, r, b = (desc.loc[k] for k in ["mouse412804", "MEArec", "rat c1", "retina"])
checks += [
    ("mouse 412804 recording present", m.subject_id == "mouse412804", f"{m.path}, subject {m.subject_id}"),
    ("mouse: 30 kHz", m.rate_hz == 30000, m.rate_hz),
    ("mouse: 27,000,000 frames (15 min)", m.frames == 27_000_000, f"{m.frames} frames, {m.duration_s:.1f} s"),
    ("mouse: 246 or 248 channels", m.channels in (246, 248), m.channels),
    ("simulated: 10 min", abs(s.duration_s - 600) < 1, f"{s.duration_s:.2f} s at {s.rate_hz:.0f} Hz"),
    ("simulated: 250 ground-truth units", s.units == 250, s.units),
    ("rat: 384 channels", r.channels == 384, r.channels),
    ("rat: 30 kHz", r.rate_hz == 30000, f"{r.rate_hz} Hz, {r.duration_s:.2f} s (notebook printed 270.0 s)"),
    ("retina: 1024 channels", b.channels == 1024, b.channels),
    ("retina: about 23199 Hz", abs(b.rate_hz - 23199) < 1, f"{b.rate_hz:.3f} Hz, {b.duration_s:.2f} s (notebook printed 135.7 s)"),
]
link = pd.DataFrame(checks, columns=["check", "holds", "value"])
print(link.to_string(index=False))
print("\nAll hold:", bool(link.holds.all()), f"({int(link.holds.sum())} of {len(link)})")

**Result: reproduced.** All 15 checks above hold: the five named files are present, the mouse 412804
recording has 248 channels (the notebook's count, not the paper's 246) at 30 kHz for 27,000,000
frames, the simulation lasts 600 s with 250 ground-truth units, the rat recording has 384 channels at
30 kHz, and the retina recording 1024 channels at 23199.09 Hz.

## Why the sorter claims are blocked

Every other claim needs the units that the six sorters (HerdingSpikes2, Kilosort2, IronClust,
Tridesclous, SpyKING Circus, HDSort) return on these recordings; the dandiset holds none of them,
and the claims file fixes the method as running those sorters. Below: what the runner offers, and
how long reading each recording whole would take at the streaming rate measured above.

In [ ]:
print("MATLAB:", env["matlab"], "| CUDA compiler:", env["nvcc"], "| GPU:", env["gpu"], "| MPI:", env["mpiexec"])
rt = pd.DataFrame(reads)
chans = desc.channels.reindex(rt.recording).values
frames_per_chunk = np.array([desc.loc[k, "chunks"][0] for k in rt.recording])
rt["chunks_read"] = rt.channels * np.ceil(rt.frames / frames_per_chunk)
rt["s_per_chunk"] = rt.read_s / rt.chunks_read
rt["chunks_whole_file"] = np.ceil(desc.frames.reindex(rt.recording).values / frames_per_chunk) * chans
rt["hours_whole_file"] = rt.chunks_whole_file * rt.s_per_chunk / 3600
print(rt.round(3).to_string(index=False))
print(f"\nShortest estimated time to stream a whole recording: {rt.hours_whole_file.min():.1f} h; notebook limit 0.5 h.")

Kilosort2, IronClust and HDSort need MATLAB, and Kilosort2 a CUDA GPU; none is on this runner and
none can be installed from a lock file. While preparing (not run here), Tridesclous 1.6.0 and
SpyKING Circus 0.9.7 installed with pip on Python 3.13, HerdingSpikes 0.3.7 failed to build, and
SpyKING Circus has no MPI to run with. Even the three Python sorters would first need each
recording read whole, which the table above puts at hours per recording.

## fig4a-curators-agree (blocked; the curator counts computed)

> The two experts' manual curations of the Kilosort2 output agree on most curated units (226 of
> 351, 64.2%), and 174 Kilosort2 units are discarded by both.

The claim depends on fig1b-unit-counts (the Kilosort2 run), which is blocked, so it is blocked for
the same reason. Its count of 174 needs the Kilosort2 run. Its other numbers come from the two
curated files alone under the criterion's second reading: `compare_two_sorters(curated 1,
curated 2)` with defaults, 226 = matched pairs, 351 = the two counts summed less the matched pairs.
They are computed here. The first reading (`compare_multiple_sorters` with the Kilosort2 run) cannot
be run.

In [ ]:
added = si.legacy_numpy_aliases()
import spikecomparison, spikeextractors
print("spikecomparison", spikecomparison.__version__, "| spikeextractors", spikeextractors.__version__, "| NumPy aliases restored:", added)
tr1 = {u: si.to_frames(t) for u, t in cur["curated 1"].items()}
tr2 = {u: si.to_frames(t) for u, t in cur["curated 2"].items()}
comp = si.compare_two(tr1, tr2)
match = comp.hungarian_match_12
n1, n2, k = len(tr1), len(tr2), int((match != -1).sum())
print(f"curated 1: {n1} units; curated 2: {n2} units")
print(f"matched pairs (both curators): {k}; units of either curator: {n1 + n2 - k}; ratio {100 * k / (n1 + n2 - k):.2f}%")
print("expected: 226 +- 22.6; 351 +- 35.1; 64.2 +- 6.42%")
scores = np.array([comp.agreement_scores.loc[a, b] for a, b in match.items() if b != -1])
print(f"agreement scores of matched pairs: min {scores.min():.3f}, median {np.median(scores):.3f}, equal to 1: {(scores == 1).sum()}")
print("\nDependence on the minimum score (criterion: the default, 0.5):")
for ms in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    c = si.compare_two(tr1, tr2, match_score=ms)
    kk = int((c.hungarian_match_12 != -1).sum())
    print(f"  match_score {ms}: both {kk}, either {n1 + n2 - kk}, {100 * kk / (n1 + n2 - kk):.2f}%")
print(f"\nrun time so far {(time.time() - started) / 60:.1f} min")

**Result: blocked (not_runnable), as the claim depends on the blocked Kilosort2 run.** The part that
does not need that run matches: the curators share 226 matched units of 351 (64.39%). These are the
paper's 226 and 351 exactly; the paper prints 64.2% for their ratio. With a minimum score of 0.3 to
0.8 the shared count ranges from 231 to 208, inside the tolerance throughout. The 174 Kilosort2 units
both curators discarded could not be counted.

## Summary

| Claim | Outcome |
|---|---|
| recordings-as-described (link) | reproduced |
| fig4a-curators-agree | blocked, not_runnable (226 of 351 curated units shared, as in the paper; the 174 needs the Kilosort2 run) |
| the 22 other claims (fig1b to fig4d) | blocked, not_runnable |

**How the analysis differs from the paper's.** No sorter was run. The paper ran six sorters, three
of them in MATLAB, on the four recordings; this attempt could only check the recordings and compare
the two curations with each other. The curations were compared with spikecomparison 0.2.6, the
version the paper's notebooks ran, on NumPy 2 with the removed NumPy aliases restored.

**What was found about the dataset.**
- The dandiset holds the four recordings and the two curations the paper names, with the stated
  rates, channel counts, durations and number of ground-truth units. It holds no sorter output, so
  the paper's comparisons can be repeated only by running the sorters again.
- The mouse recording has 248 channels; the paper says 246 active channels. Which two are inactive
  is not marked.
- The curated sortings carry no rate, no unit labels and no link to the Kilosort2 units they were
  curated from; their spike times are consistent with the mouse recording's clock (trough at lag 0).
- The retina recording is stored as unsigned counts centred near 2048 with a stated conversion of
  1e-6 V and no offset, which would give a DC level of about 2 mV; the gain and offset of the
  Biocam's converter are not in the file.
- Every raw series is chunked one channel per chunk, so streaming a stretch of all channels is
  very slow (table above).
- The paper's 64.2% for 226 of 351 is 64.39%.